# Nemotron 3.5 Transcribe 30B-A4B — 40-language ASR on a free T4 (best-effort 4-bit)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/nemotron-35-transcribe-t4-f78a/colab/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B.ipynb)

[`nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B) is NVIDIA's offline speech-augmented language model: **~34B MoE / ~4.3B active** (including the audio encoder). It does multilingual transcription in **40 languages**, speaker-attributed transcripts, speech translation, contextual prompting, and audio question answering. Architecture: Mamba-2–Transformer hybrid MoE language backbone plus a Parallel Expert Encoder (Transformer ASR encoder + Nemotron-3-Diarization speaker encoder). Context is 65,536 tokens; recordings up to one hour. Native Transformers class: `AutoModelForMultimodalLM` / `Nemotron3_5TranscribeProcessor`.

Why this model: one open-weight checkpoint covers transcription, who-spoke-when tags, translation, and audio QA. The [Granite Speech notebook](granite-speech-5.0-2b-asr-translation.ipynb) is a 2.3B ASR/translation walkthrough that actually fits a T4. The [Nemotron-Labs Unified ASR 0.6B notebook](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb) is a small FastConformer. [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb) writes speaker timelines only. This notebook is the large SALM.

**Licence:** [OpenMDW-1.1](https://openmdw.ai/license/1-1/). Commercial and non-commercial use are allowed. If you redistribute the weights or card artifacts, keep the license text and origin notices. Patent/copyright litigation against a provider terminates the grant. Generated outputs are unrestricted. The materials are provided as-is; you are responsible for clearing third-party rights in the audio you run. **Not gated** — no Hugging Face login.

**Official hardware (not this notebook):** the card's deployment target is **1× H100 80 GB** with vLLM 0.32 + NeMo Speech. The hardware table lists A100 80 GB BF16, L40S FP8, Blackwell NVFP4, and similar 48–80 GB cards. A free Colab T4 (~15 GB, Turing sm_75) is **not** on that list.

**Quant / T4 path (what this notebook actually loads):**

| Variant | Hub id | On-disk | Can Transformers load it on a T4? |
| --- | --- | --- | --- |
| BF16 (this notebook's source) | [`nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B) | **67.5 GB** `model.safetensors` | Not in BF16/FP16. Loaded here as **bitsandbytes 4-bit NF4 + float16 compute**. |
| FP8 (ModelOpt) | [`…-A4B-FP8`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-FP8) | ~34 GB | **No.** Card hardware is L40S / H100 / Blackwell. T4 has no FP8 tensor cores. |
| NVFP4 (ModelOpt mixed) | [`…-A4B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-NVFP4) | ~21 GB | **No.** Card says **Blackwell GPUs only**. |

bitsandbytes 4-bit is the lowest *documented* Transformers quant that can theoretically run on a T4 (it rewrites `nn.Linear`, including MoE experts). Mamba conv / SSM states stay in float16. There is **no official 4-bit export** and **no documented bitsandbytes recipe** for this architecture — if load fails with an unsupported-module error, that is the end of the T4 path. This notebook does **not** invent a second quant.

**VRAM / disk assumption (please read):** *Runtime → Change runtime type → T4 GPU*. 33.7B × 0.5 bytes ≈ **16.8 GB** of packed 4-bit Linear weights, plus leftover float16 modules (audio encoder ~0.8B, Mamba convs, norms). That is already **at or above** the free T4's ~15 GB. `device_map="auto"` leaves ~2 GB of GPU headroom and spills overflow to CPU RAM. **Expected outcome on free Colab: OOM or host-RAM / disk failure is likely.** A realistic 4-bit box is a **40 GB A100**; the official path is **H100 80 GB**. First run also downloads the **67.5 GB** BF16 shard (bitsandbytes quantizes it while loading). Free Colab disk is often ~78 GB — the download can fill the disk before the GPU is involved. Short clips only (`max_new_tokens` 256–512).

**Not in this notebook:** vLLM + NeMo Speech serving, SGLang-Omni, word-timestamp / CTC alignment (`torchaudio` + `scipy`), the official FP8 / NVFP4 checkpoints, FlashAttention-2, or hour-long recordings.

**Links:** [model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B) · [Transformers docs](https://huggingface.co/docs/transformers/model_doc/nemotron3_5_transcribe) · [tech report](https://research.nvidia.com/labs/nemotron/files/NVIDIA-Nemotron-3.5-Transcribe-Technical-Report.pdf) · [NeMo Speech](https://github.com/NVIDIA-NeMo/Speech) · [demo](https://huggingface.co/spaces/nvidia/nemotron-transcribe) · [OpenMDW-1.1](https://openmdw.ai/license/1-1/) · [announcement](https://x.com/NVIDIAAI/status/2115462300241141882)


## 1. Install Transformers from source + bitsandbytes

The card's Transformers path is `pip install git+https://github.com/huggingface/transformers` (`AutoModelForMultimodalLM` / `Nemotron3_5TranscribeProcessor` landed around **5.22.0.dev0**). Colab's preinstalled wheel is older and does not know this class.

`bitsandbytes` is the T4 load path, not a fallback: BF16 (67.5 GB) and float16 copies do not fit ~15 GB. `accelerate` is required for `device_map="auto"`. Keep Colab's torch. Do **not** install `flash-attn`, vLLM, or NeMo Speech. `soundfile` decodes the short sample clips.


In [ ]:
%pip install -q -U "git+https://github.com/huggingface/transformers" "accelerate>=1.2.0" "bitsandbytes>=0.46.0" soundfile

# Card Quick Start: transformers from source (AutoModelForMultimodalLM).
# bitsandbytes 4-bit NF4 is the T4 path; official FP8/NVFP4 need L40S/H100/Blackwell.
# accelerate is required for device_map="auto" / CPU spill.
# Do NOT replace Colab's torch. Do NOT install flash-attn, vLLM, or nemo_toolkit.


> **Restart the runtime now:** *Runtime → Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The git install is invisible until the process restarts.


## 2. Check the GPU and print T4 caveats

The next cell must see a CUDA GPU. It prints `torch.cuda.is_available()`, the device name, and free VRAM, then stops if you are on CPU.


In [ ]:
import gc
import shutil

import torch
import transformers
from packaging.version import Version

# Nemotron 3.5 Transcribe's Transformers class is in 5.22.dev / later source builds.
_tf_ver = Version(transformers.__version__.split("+")[0].split(".dev")[0])
assert _tf_ver >= Version("5.22.0"), (
    f"transformers {transformers.__version__} is loaded; this model needs 5.22+ "
    "(card: pip install git+https://github.com/huggingface/transformers). "
    "Run the install cell, then Runtime → Restart session."
)
from transformers import AutoModelForMultimodalLM  # noqa: F401

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime → Change runtime type → T4 GPU, then rerun from the top.")

# T4 (sm_75): no FlashAttention-2, no native bfloat16 tensor cores, no FP8 / NVFP4.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8
free, total = torch.cuda.mem_get_info()
disk = shutil.disk_usage("/")
DTYPE = torch.float16  # 4-bit compute dtype; T4 has no native bf16

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free / 1024**3:.1f} GB of {total / 1024**3:.1f} GB")
print(f"disk free: {disk.free / 1024**3:.1f} GB (need ~68 GB for the BF16 shard)")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: 4-bit NF4 + float16 compute  (BF16 67.5 GB / FP8 / NVFP4 skipped)")
if disk.free < 70 * 1024**3:
    print("WARNING: disk is tight. The 67.5 GB download may fail on free Colab.")
if total < 20 * 1024**3:
    print("WARNING: this GPU is under 20 GB. Packed 4-bit of 33.7B is ~16.8 GB — T4 OOM is likely.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load the BF16 checkpoint in 4-bit NF4

`AutoProcessor` + `AutoModelForMultimodalLM`, matching the [card's Transformers snippet](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B#transformers-usage-) (typo on some copies: `AutoProcess or` → `AutoProcessor`). No `trust_remote_code`.

The first run downloads **67.5 GB**, then packs Linear / MoE expert weights to 4-bit NF4 on the fly. `device_map="auto"` with a GPU budget of `total − 2 GB` spills whatever does not fit to CPU. That is still experimental on a 15 GB T4 + ~12 GB host RAM.

Official FP8 / NVFP4 Hub ids were checked and **not** used: Transformers cannot run those ModelOpt checkpoints on Turing.


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B"
SAMPLES_REPO = "hf-internal-testing/dummy-audio-samples"
MAX_NEW_TOKENS_ASR = 256
MAX_NEW_TOKENS_SPK = 512
MAX_NEW_TOKENS_QA = 256

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Leave ~2 GB of GPU headroom for the audio encoder activations + KV cache.
# Packed 4-bit of 33.7B is ~16.8 GB, so a 15 GB T4 will spill to CPU.
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_budget = max(8.0, total_gb - 2.0)
max_memory = {0: f"{gpu_budget:.1f}GiB", "cpu": "22GiB"}

print("MODEL_ID:", MODEL_ID)
print("quant: 4-bit NF4 + float16 compute | double_quant:", True)
print("max_memory:", max_memory)
print("Downloading / packing 67.5 GB BF16 → 4-bit (first run is slow)...")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("processor:", type(processor).__name__)

try:
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        dtype=DTYPE,
    ).eval()
except Exception as exc:
    raise RuntimeError(
        "4-bit NF4 load failed. Official FP8/NVFP4 checkpoints are not T4-loadable "
        "(L40S/H100/Blackwell + ModelOpt). There is no other documented Transformers "
        "quant for this hybrid Mamba-2 / MoE SALM. Use an H100 80 GB (vLLM) or a "
        "40 GB+ GPU if you want to continue."
    ) from exc

hf_dev = getattr(model, "hf_device_map", None)
if hf_dev:
    from collections import Counter

    counts = Counter(str(v) for v in hf_dev.values())
    print("device_map:", dict(counts))
print("class:", type(model).__name__)
print("dtype (config):", getattr(model.config, "dtype", None), "| compute:", DTYPE)
print_vram("after 4-bit load")


## 4. Short public sample clips

Two card-documented fixtures from [`hf-internal-testing/dummy-audio-samples`](https://huggingface.co/datasets/hf-internal-testing/dummy-audio-samples):

1. `en-Alice_woman.wav` (~297 KB) — plain English transcription and the audio-QA demo
2. `diarization_example.mp3` (~293 KB) — speaker-attributed transcription (card prompt)

Keep clips this short. Do not paste a meeting recording into a T4 session.


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

SPEAKER_PROMPT = (
    "Transcribe this audio with punctuation and capitalization. Include all spoken "
    "words and verbal fillers. Write numbers in their standard written form. Put "
    "speaker tags <spk:0> <spk:1> <spk:2> <spk:3> on the transcription."
)
QA_SYSTEM = (
    "Answer the question directly using evidence from every provided audio clip in "
    "order. Answer every requested part, preserve exact names and numbers, and give "
    "a concise, specific answer first. Do not invent details."
)


def fetch_clip(name):
    path = Path(
        hf_hub_download(
            repo_id=SAMPLES_REPO,
            repo_type="dataset",
            filename=name,
        )
    )
    print(f"{name}: {path.stat().st_size / 1024:.0f} KB → {path}")
    return path


alice_path = fetch_clip("en-Alice_woman.wav")
diar_path = fetch_clip("diarization_example.mp3")


def move_to_model(inputs):
    # device_map="auto" may span GPU + CPU; send the batch to the first module device.
    target = getattr(model, "device", None) or DEVICE
    return inputs.to(target)


def generate_text(inputs, max_new_tokens, *, thinking=False, skip_special_tokens=True):
    # Greedy for ASR; card sampling (temp 0.6 / top_p 0.95) when thinking is on.
    gen_kwargs = dict(max_new_tokens=max_new_tokens)
    if thinking:
        gen_kwargs.update(do_sample=True, temperature=0.6, top_p=0.95)
    else:
        gen_kwargs.update(do_sample=False)
    prompt_len = inputs["input_ids"].shape[1]
    print(f"prompt tokens: {prompt_len} | max_new_tokens: {max_new_tokens} | thinking: {thinking}")
    with torch.inference_mode():
        outputs = model.generate(**inputs, **gen_kwargs)
    sequences = outputs.sequences if hasattr(outputs, "sequences") else outputs
    text = processor.decode(sequences[0, prompt_len:], skip_special_tokens=skip_special_tokens)
    pad = getattr(processor.tokenizer, "pad_token", None)
    if pad:
        text = text.replace(pad, "")
    return text.strip()


## 5. Demo 1 — plain English transcription

Card Quick Start: `processor.apply_transcription_request(...)` on `en-Alice_woman.wav`, default verbatim prompt, `max_new_tokens=256`, thinking off, greedy decode.


In [ ]:
print("Demo 1 clip:")
display(Audio(filename=str(alice_path)))

inputs = move_to_model(processor.apply_transcription_request(str(alice_path)))
text = generate_text(inputs, MAX_NEW_TOKENS_ASR, thinking=False, skip_special_tokens=True)
print()
print(text)
print()
print_vram("after demo 1")
del inputs
gc.collect()
torch.cuda.empty_cache()


## 6. Demo 2 — speaker-attributed transcription

Card speaker-tag prompt on `diarization_example.mp3` via `apply_chat_template(..., enable_thinking=False)`. `skip_special_tokens=False` keeps `<spk:N>` tags. `processor.extract_speaker_dict` splits turns. `max_new_tokens=512` (card uses 1024; keep it modest on a T4).


In [ ]:
print("Demo 2 clip:")
display(Audio(filename=str(diar_path)))

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": SPEAKER_PROMPT},
            {"type": "audio", "path": str(diar_path)},
        ],
    }
]
inputs = move_to_model(
    processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        enable_thinking=False,
    )
)
text = generate_text(inputs, MAX_NEW_TOKENS_SPK, thinking=False, skip_special_tokens=False)
print()
print(text)
print()
if hasattr(processor, "extract_speaker_dict"):
    print("speaker turns:")
    for turn in processor.extract_speaker_dict(text):
        print(turn)
print()
print_vram("after demo 2")
del inputs
gc.collect()
torch.cuda.empty_cache()


## 7. Demo 3 — short audio QA with `enable_thinking`

Card audio-understanding preset: system prompt from the Prompt Guide, user question `What is happening in this audio?`, `enable_thinking=True`, temperature 0.6 / top_p 0.95. Completion is capped at **256** new tokens (the card's 20,480 / 16,384 reasoning budget will not fit a T4). Same short Alice clip.


In [ ]:
print("Demo 3 clip (same Alice WAV, audio QA):")
display(Audio(filename=str(alice_path)))

conversation = [
    {"role": "system", "content": QA_SYSTEM},
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What is happening in this audio?"},
            {"type": "audio", "path": str(alice_path)},
        ],
    },
]
inputs = move_to_model(
    processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        enable_thinking=True,
    )
)
text = generate_text(inputs, MAX_NEW_TOKENS_QA, thinking=True, skip_special_tokens=False)
print()
print(text)
print()
print_vram("after demo 3")
del inputs
gc.collect()
torch.cuda.empty_cache()


## 8. Caveats and cleanup

**What this notebook is for.** A best-effort Transformers smoke test of Nemotron 3.5 Transcribe on free Colab: 4-bit NF4, three short public clips (plain ASR, speaker tags, audio QA with thinking).

**What it is not.** The official serving stack (vLLM 0.32 + NeMo Speech on 1× H100 80 GB), the ModelOpt [FP8](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-FP8) or [NVFP4](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B-NVFP4) checkpoints, word timestamps, or long-form / 40-language coverage testing.

**T4 expectation.** Packed 4-bit of a 33.7B MoE is ~16.8 GB before activations. A 15 GB T4 will often OOM on load or on the first `generate`. Host RAM (~12 GB) and disk (~78 GB vs a 67.5 GB download) are the other two failure modes. Prefer a 40 GB A100 for this 4-bit path, or H100 80 GB for the card's BF16 / vLLM path.

**Related notebooks:** [Granite Speech 5.0 2B](granite-speech-5.0-2b-asr-translation.ipynb) (fits a T4), [Nemotron-Labs Unified ASR 0.6B](Nemotron-Labs-Unified-ASR-0.6B-multilingual.ipynb), [Nemotron-3 Diarization](Nemotron-3-Diarization.ipynb), [Whisper](AIBackends-Whisper-audio-transcription.ipynb).

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
for name in ("model", "processor", "inputs"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram("after cleanup")
print("Released. Re-run the load cell if you want another clip.")
